# Thesis LLM server on Colab: Qwen3.8-27B Uncensored, maximum context

Running this notebook gives you an **OpenAI-compatible model server inside this Colab machine**, using the setup we tested:

| | |
|---|---|
| Model | `orcarouter/Qwen3.8-27B-Uncensored-GGUF`, an abliterated Qwen3.8-27B (rated first of 12 uncensored versions in the independent Abliterlitics comparison) |
| Endpoint | **`http://localhost:1234/v1`** (same address as LM Studio). Model name `qwen38-unc`; API key: any string |
| Context window | **Auto-tuned**: the largest size that stays **100% on the GPU** (up to the model's native 262,144) |
| Context memory | 8-bit (`q8_0`): roughly halves the memory each token needs (measured working) |
| Quantization | `auto`: **Q4_K_M on an L4**, **Q8_0 on an A100** |

**Verified on 2026-11-10 (L4):**
- Q4_K_M at a **131,072**-token context: 100% GPU, 20,580 of 23,034 MiB.
- About **13.8 tokens/s**.
- The model download from Hugging Face took **66 s**.

On an L4, the auto-tuner may settle a little higher than 131,072, with less spare memory. Run the long-context test (step 9) once to confirm. The A100 path follows the same logic but hasn't been tested yet.

### Before the first run (one time)
1. Sign in at huggingface.co and open https://huggingface.co/orcarouter/Qwen3.8-27B-Uncensored-GGUF. **Accept its terms**: the repo is gated.
2. Create a **read token** at https://huggingface.co/settings/tokens.
3. *(Optional, but it saves typing every session)* In Colab, open the **key icon** in the left sidebar, add a secret named **`HF_TOKEN`** with the token, and enable notebook access. Without it, step 4 asks you to log in with a code.

### Every session
1. **Runtime → Change runtime type → L4 GPU** (or A100 if offered) → Save.
2. **Runtime → Run all.** The setup takes about 5–8 minutes. An L4 costs about **1.71 compute units/hour**; an A100 40GB about 5.40.
3. Your pipeline uses `base_url="http://localhost:1234/v1"`, `model="qwen38-unc"`, `api_key="ollama"`. The pipeline must run **on this machine**: see step 10.
4. When finished: **Runtime → Disconnect and delete runtime.** Billing runs for as long as a GPU is connected, even if idle.

## 0. Settings: the only cell you normally edit

In [ ]:
HF_REPO        = "orcarouter/Qwen3.8-27B-Uncensored-GGUF"  # gated: accept the terms on its HF page first
QUANT          = "auto"        # "auto" = Q4_K_M on L4, Q8_0 on A100. Or force e.g. "Q4_K_M", "Q6_K", "Q8_0", "IQ4_XS"
CONTEXT_LENGTH = "auto"        # "auto" = largest context that stays 100% on the GPU. Or a number, e.g. 131072 (verified on L4)
MODEL          = "qwen38-unc"  # the "model" name your pipeline sends
PORT           = 1234          # same port as LM Studio -> http://localhost:1234/v1
KV_CACHE_8BIT  = True          # 8-bit context memory: ~half the memory per token (verified with this model)
KEEP_ALIVE     = "60m"         # keep the model loaded between requests
MIN_FREE_MIB   = 1000          # spare GPU memory the auto-tuner must leave (safety margin)
DRIVE_DIR      = "/content/drive/MyDrive/thesis_llm"  # results, logs, chats, run config (NOT the model)

RUN_LONG_CONTEXT_TEST = False  # step 9: fills ~90% of the context to prove it really works (takes several minutes)
RUN_CHAT              = False  # step 11: open the interactive chat at the end of "Run all"

## 1. Check the GPU and choose the model file

In [ ]:
import subprocess, os, time, json, datetime

def _smi(query):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip().splitlines()
    except FileNotFoundError:
        return ""
    return out[0].strip() if out else ""

def gpu_mem_used_mib():
    try:
        return int(float(_smi("memory.used")))
    except ValueError:
        return 0

def pick_quant(total_mib, requested):
    if requested != "auto":
        return requested
    if total_mib >= 38000:   # A100 40GB / 80GB
        return "Q8_0"
    if total_mib >= 20000:   # L4
        return "Q4_K_M"
    return "IQ3_M"           # small GPUs (e.g. T4): reduced quality

GPU_NAME = _smi("name") or "none"
GPU_TOTAL_MIB = int(float(_smi("memory.total") or 0))
if GPU_TOTAL_MIB == 0:
    raise RuntimeError("No GPU found. Runtime -> Change runtime type -> L4 GPU (or A100), then Run all again.")
QUANT_SEL = pick_quant(GPU_TOTAL_MIB, QUANT)
print(f"GPU: {GPU_NAME} | {GPU_TOTAL_MIB} MiB | model file: {QUANT_SEL}")
if GPU_TOTAL_MIB < 20000:
    print("WARNING: small GPU (e.g. T4). Using a 3-bit file with reduced quality. Switch to an L4 for real work.")

## 2. Install Ollama and the Python clients

In [ ]:
!apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 3
!pip -q install ollama openai

## 3. Connect Google Drive (for results only)

The model is **not** stored on Drive: loading it from Drive took 327 s, versus a 66 s fresh download. Drive holds your logs, results, chats and the run configuration.

If mounting fails with *"credential propagation was unsuccessful"*:
- In the popup, pick the **same Google account** you're using in Colab, and tick all permissions.
- Or use the **folder icon → Mount Drive** button in the left sidebar.

The notebook continues anyway, but results are then saved on the machine and are lost when it shuts down.

In [ ]:
OUT_DIR = "/content/thesis_llm"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = DRIVE_DIR
except Exception as e:
    print(f"WARNING: Drive not mounted ({e}). Results go to {OUT_DIR} and are LOST when the runtime ends.")
os.makedirs(OUT_DIR, exist_ok=True)
print("Results folder:", OUT_DIR)

## 4. Log in to Hugging Face and download the model

The download goes to the fast local disk. It took about 66 s for the 16.8 GB Q4_K_M file; Q8_0 is larger.

In [ ]:
from huggingface_hub import login, list_repo_files, hf_hub_download

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass
if token:
    login(token=token)
else:
    login()   # shows a link and a code to authorize

def pick_gguf(files, quant):
    suffix = f"-{quant.lower()}.gguf"
    return [f for f in files if f.lower().endswith(suffix) and "mmproj" not in f.lower()]

repo_files = list_repo_files(HF_REPO)
matches = pick_gguf(repo_files, QUANT_SEL)
if not matches:
    raise RuntimeError(f"No {QUANT_SEL} file in {HF_REPO}. Available: {[f for f in repo_files if f.endswith('.gguf')]}")
GGUF_FILE = matches[0]

t = time.time()
GGUF_PATH = hf_hub_download(HF_REPO, GGUF_FILE, local_dir="/content/models")
FILE_MIB = os.path.getsize(GGUF_PATH) / 2**20
print(f"Downloaded {GGUF_FILE} ({FILE_MIB/1024:.1f} GiB) in {time.time()-t:.0f}s")

## 5. Server helpers

These functions start, stop and test the Ollama server.

- **Port 1234** means LM Studio-style clients work unchanged.
- When restarting, the helpers **wait until the GPU memory is actually freed**. Otherwise the new server would see the old model still occupying the GPU and wrongly push part of the model onto the CPU.

In [ ]:
import requests, ollama

OLLAMA_LOG = "/content/ollama.log"
SERVER = None

def stop_server(timeout=120):
    subprocess.run(["pkill", "-f", "ollama (serve|runner)"])
    for _ in range(timeout):
        if gpu_mem_used_mib() < 1000:
            return
        time.sleep(1)
    print("WARNING: GPU memory was not fully released")

def start_server(ctx):
    global SERVER
    stop_server()
    os.environ.update({
        "OLLAMA_MODELS": "/content/ollama_models",      # local disk (fast), not Drive
        "OLLAMA_HOST": f"127.0.0.1:{PORT}",
        "OLLAMA_CONTEXT_LENGTH": str(ctx),
        "OLLAMA_KEEP_ALIVE": KEEP_ALIVE,
    })
    if KV_CACHE_8BIT:
        os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
        os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
    else:
        os.environ.pop("OLLAMA_KV_CACHE_TYPE", None)
    SERVER = subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "w"), stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            requests.get(f"http://127.0.0.1:{PORT}", timeout=2)
            return
        except requests.exceptions.RequestException:
            time.sleep(1)
    raise RuntimeError(f"Ollama did not start - see {OLLAMA_LOG}")

def ollama_ps():
    return subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout

client = ollama.Client(host=f"http://127.0.0.1:{PORT}")

def load_and_check(ctx):
    """Restart the server with this context, load the model, check it is 100% on the GPU."""
    start_server(ctx)
    t = time.time()
    try:
        client.chat(model=MODEL, messages=[{"role": "user", "content": "Say OK."}],
                    think=False, options={"num_predict": 8})
    except Exception as e:
        return False, {"ctx": ctx, "error": str(e)[:300]}
    ps = ollama_ps()
    used = gpu_mem_used_mib()
    info = {"ctx": ctx, "load_s": round(time.time() - t), "gpu_used_mib": used,
            "free_mib": GPU_TOTAL_MIB - used, "full_gpu": "100% GPU" in ps}
    info["ok"] = info["full_gpu"] and info["free_mib"] >= MIN_FREE_MIB
    return info["ok"], info

## 6. Register the model with Ollama

The model's default sampling settings are Qwen's recommendations for thinking mode: temperature 1.0, top_p 0.95, top_k 20, min_p 0. Setting them here means every client gets them, including the `/v1` endpoint, which has no `top_k` parameter. Your requests can still override temperature and top_p.

In [ ]:
start_server(32768)
open("/content/Modelfile", "w").write(
    f"FROM {GGUF_PATH}\n"
    "PARAMETER temperature 1.0\n"
    "PARAMETER top_p 0.95\n"
    "PARAMETER top_k 20\n"
    "PARAMETER min_p 0\n"
)
!ollama create {MODEL} -f /content/Modelfile
!ollama list
OLLAMA_VERSION = subprocess.run(["ollama", "-v"], capture_output=True, text=True).stdout.strip()
print(OLLAMA_VERSION)

## 7. Find the largest context that stays 100% on the GPU

This step tries sizes from large to small in steps of 16,384 tokens, capped at the model's native 262,144. It starts at the largest size that a memory estimate says could fit; the estimate is based on what we measured, about 0.038 MiB per token with the 8-bit cache. For each size it loads the model and checks two things:
- **100% GPU**, so no part of the model is pushed onto the CPU;
- at least `MIN_FREE_MIB` of GPU memory left free.

It keeps the first size that passes. Each try takes about 20–60 s.

Expected results:
- **L4:** about 131,072–163,840 (131,072 is the verified size).
- **A100 40GB/80GB:** about 262,144.

In [ ]:
KV_MIB_PER_TOKEN = 0.0381 if KV_CACHE_8BIT else 0.072   # measured on the L4 with the 8-bit cache, 2026-11-10
NATIVE_MAX_CTX = 262144

def estimate_used_mib(ctx):
    return FILE_MIB - 250 + ctx * KV_MIB_PER_TOKEN + 50

def context_candidates():
    if CONTEXT_LENGTH != "auto":
        return [int(CONTEXT_LENGTH)]
    cands = list(range(NATIVE_MAX_CTX, 32767, -16384))
    fitting = [c for c in cands if GPU_TOTAL_MIB - estimate_used_mib(c) >= MIN_FREE_MIB / 2]
    return fitting or [32768]

ATTEMPTS, CTX = [], None
for ctx in context_candidates():
    print(f"Trying context {ctx:,} ...", end=" ", flush=True)
    ok, info = load_and_check(ctx)
    ATTEMPTS.append(info)
    print("OK" if ok else "does not fit", info)
    if ok:
        CTX = ctx
        break
if CTX is None:
    raise RuntimeError(f"No context size fitted 100% on the GPU. See the attempts above and {OLLAMA_LOG}.")

RUN_CONFIG = {
    "time": datetime.datetime.now().isoformat(timespec="seconds"),
    "gpu": GPU_NAME, "gpu_total_mib": GPU_TOTAL_MIB, "gpu_used_mib": ATTEMPTS[-1]["gpu_used_mib"],
    "hf_repo": HF_REPO, "gguf_file": GGUF_FILE, "quant": QUANT_SEL, "file_mib": round(FILE_MIB),
    "context_length": CTX, "kv_cache": "q8_0" if KV_CACHE_8BIT else "f16", "keep_alive": KEEP_ALIVE,
    "endpoint": f"http://localhost:{PORT}/v1", "model": MODEL, "ollama_version": OLLAMA_VERSION,
    "sampling_defaults": {"temperature": 1.0, "top_p": 0.95, "top_k": 20, "min_p": 0},
    "autotune_attempts": ATTEMPTS,
}
json.dump(RUN_CONFIG, open(os.path.join(OUT_DIR, "run_config.json"), "w"), indent=1)

print(f"""
================= MODEL SERVER READY =================
Endpoint (OpenAI-compatible): http://localhost:{PORT}/v1
Model name: {MODEL}      API key: any string, e.g. "ollama"
Weights:    {HF_REPO} / {GGUF_FILE}
GPU:        {GPU_NAME}: {ATTEMPTS[-1]['gpu_used_mib']:,} of {GPU_TOTAL_MIB:,} MiB used
CONTEXT:    {CTX:,} tokens (100% GPU), context memory {RUN_CONFIG['kv_cache']}
Run config saved to {os.path.join(OUT_DIR, 'run_config.json')}
=======================================================""")
print(ollama_ps())

## 8. Quick checks: speed, the `/v1` endpoint, structured output

**8a. Speed check** (thinking off).

In [ ]:
t = time.time()
r = client.chat(model=MODEL, think=False,
                messages=[{"role": "user", "content": "Write a short Python function that checks whether a string is a palindrome."}],
                options={"num_predict": 512})
print(r.message.content)
print(f"\n{time.time()-t:.0f}s | prompt {r.prompt_eval_count} | output {r.eval_count} | "
      f"{r.eval_count/(r.eval_duration/1e9):.1f} tok/s | done_reason {r.done_reason}")

**8b. Pipeline helpers.** These are the functions to copy into your pipeline:
- `llm_call()`: calls the `/v1` endpoint, logs every call to `llm_calls.jsonl` on Drive, and **warns** if the prompt was probably cut off at the start (Ollama does this silently), if the output was cut off, or if the answer is empty.
- `generate_with_retries()`: the **smart retry**. Each retry sends only the original task, the **last** failed output and its error. The context stays constant, and thinking is never sent back.

In [ ]:
from openai import OpenAI

llm = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="ollama")
LOG_PATH = os.path.join(OUT_DIR, "llm_calls.jsonl")
CHARS_PER_TOKEN = 3.5          # rough; step 9 measures the real value
os.environ["LLM_BASE_URL"] = f"http://localhost:{PORT}/v1"   # for your pipeline (step 10)
os.environ["LLM_MODEL"] = MODEL

def llm_call(messages, schema=None, max_tokens=None, temperature=1.0, top_p=0.95, tag=""):
    """One call to the /v1 endpoint. Returns (content, reasoning, record). Every call is logged."""
    if max_tokens is None:
        max_tokens = min(49152, CTX // 2)
    est_prompt = sum(len(m["content"]) for m in messages) / CHARS_PER_TOKEN
    if est_prompt + max_tokens > CTX:
        print(f"WARNING: estimated prompt ({est_prompt:,.0f}) + max_tokens ({max_tokens:,}) > context ({CTX:,})")
    kwargs = dict(model=MODEL, messages=messages, max_tokens=max_tokens, temperature=temperature, top_p=top_p)
    if schema is not None:
        kwargs["response_format"] = {"type": "json_schema", "json_schema": {"name": "output", "schema": schema}}
    t = time.time()
    r = llm.chat.completions.create(**kwargs)
    choice = r.choices[0]
    reasoning = (choice.message.model_extra or {}).get("reasoning") or ""
    content = choice.message.content or ""
    rec = {"time": datetime.datetime.now().isoformat(timespec="seconds"), "tag": tag,
           "seconds": round(time.time() - t, 1), "prompt_tokens": r.usage.prompt_tokens,
           "completion_tokens": r.usage.completion_tokens, "finish_reason": choice.finish_reason,
           "max_tokens": max_tokens, "context": CTX, "model": MODEL, "quant": QUANT_SEL, "gpu": GPU_NAME,
           "reasoning_chars": len(reasoning), "content_chars": len(content), "warnings": []}
    if r.usage.prompt_tokens >= 0.95 * CTX:
        rec["warnings"].append("PROMPT PROBABLY TRUNCATED: prompt_tokens is close to the context size")
    if choice.finish_reason == "length":
        rec["warnings"].append("OUTPUT CUT OFF (finish_reason=length): raise max_tokens or shorten the prompt")
    if not content.strip():
        rec["warnings"].append("EMPTY ANSWER: thinking may have used the whole budget")
    with open(LOG_PATH, "a") as f:
        f.write(json.dumps(rec) + "\n")
    for w in rec["warnings"]:
        print("WARNING:", w)
    return content, reasoning, rec

def retry_messages(system, task, last_output, error):
    return [{"role": "system", "content": system},
            {"role": "user", "content": task},
            {"role": "assistant", "content": last_output},          # content only, never the thinking
            {"role": "user", "content": f"Your previous output failed validation:\n{error}\n"
                                        "Return the FULL corrected output, not just the changes."}]

def generate_with_retries(system, task, validate, schema=None, max_retries=5, tag="task", **call_kwargs):
    """validate(text) -> (ok: bool, error: str). Returns the first valid output."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    error = ""
    for attempt in range(max_retries + 1):
        content, reasoning, rec = llm_call(messages, schema=schema, tag=f"{tag}#{attempt}", **call_kwargs)
        ok, error = validate(content)
        print(f"{tag} attempt {attempt}: {'OK' if ok else 'FAILED'} | prompt {rec['prompt_tokens']:,} "
              f"| output {rec['completion_tokens']:,} | {rec['seconds']}s")
        if ok:
            return content
        messages = retry_messages(system, task, content, error)
    raise RuntimeError(f"{tag}: still failing after {max_retries} retries. Last error: {error}")

def pydantic_validator(model_cls):
    def validate(text):
        try:
            model_cls.model_validate_json(text)
            return True, ""
        except Exception as e:
            return False, str(e)[:4000]
    return validate

print("Helpers ready. Calls are logged to", LOG_PATH)

**8c. Thinking through the `/v1` endpoint.** The reasoning comes back in a separate `reasoning` field; `content` is just the answer.

In [ ]:
content, reasoning, rec = llm_call(
    [{"role": "user", "content": "Plan a tiny Python CLI todo app: list the files and functions you would create. Be brief."}],
    max_tokens=8192, tag="endpoint-test")
print(content)
print(f"\nreasoning: {len(reasoning):,} chars | prompt {rec['prompt_tokens']} | output {rec['completion_tokens']} "
      f"| {rec['seconds']}s | finish_reason {rec['finish_reason']}")

**8d. JSON-schema output with thinking on.** This checks the open question of whether the schema is enforced while the model thinks. Replace `CodePlan` with your thesis schema later.

In [ ]:
from pydantic import BaseModel
from typing import List

class Function(BaseModel):
    name: str
    signature: str
    purpose: str

class File(BaseModel):
    path: str
    purpose: str
    functions: List[Function]

class CodePlan(BaseModel):
    project_name: str
    libraries: List[str]
    files: List[File]

plan_json = generate_with_retries(
    system="You are a senior software architect. Output only the code plan as JSON.",
    task="Design a code plan for a small Python CLI todo app that stores tasks in a JSON file.",
    validate=pydantic_validator(CodePlan), schema=CodePlan.model_json_schema(),
    max_retries=2, tag="schema-test", max_tokens=8192)
plan = CodePlan.model_validate_json(plan_json)
print(f"\nValid CodePlan: {plan.project_name} | {len(plan.files)} files | libraries: {plan.libraries}")

## 9. (Optional) Long-context test

This proves the chosen context really works with a long input. It plants a secret code at the **very start** of a prompt that fills about 90% of the context, then asks for the code at the end. If Ollama had silently cut off the beginning, the answer would be wrong.

It also measures **prompt reading speed** (tokens/s), which tells you how long your big prompts take before the model starts writing. On an L4 this takes several minutes. Set `RUN_LONG_CONTEXT_TEST = True` in step 0, or call `long_context_test()` manually.

In [ ]:
def long_context_test(fill=0.9):
    global CHARS_PER_TOKEN
    unit = ("def process_record(record):\n"
            "    # Validate, normalise and store one record.\n"
            "    if not record.get('id'):\n"
            "        raise ValueError('missing id')\n"
            "    return {k: str(v).strip() for k, v in record.items()}\n\n")
    sample = unit * 20
    r = client.chat(model=MODEL, messages=[{"role": "user", "content": sample}], think=False,
                    options={"num_predict": 1})
    CHARS_PER_TOKEN = len(sample) / r.prompt_eval_count
    needle = "IMPORTANT: the secret build code is ZEBRA-7341. Remember it.\n\n"
    question = "\n\nWhat is the secret build code mentioned at the very beginning? Answer with the code only."
    target_tokens = int(CTX * fill)
    n_units = max(1, int((target_tokens * CHARS_PER_TOKEN - len(needle) - len(question)) / len(unit)))
    prompt = needle + unit * n_units + question
    print(f"Sending ~{target_tokens:,} tokens ({fill:.0%} of {CTX:,}); measured {CHARS_PER_TOKEN:.2f} chars/token ...")
    t = time.time()
    r = client.chat(model=MODEL, messages=[{"role": "user", "content": prompt}], think=False,
                    options={"num_predict": 32})
    answer = r.message.content.strip()
    passed = "ZEBRA-7341" in answer
    print(f"Prompt tokens read: {r.prompt_eval_count:,} | prompt reading: "
          f"{r.prompt_eval_count/(r.prompt_eval_duration/1e9):,.0f} tok/s | total {time.time()-t:.0f}s")
    print(f"Answer: {answer!r} -> {'PASS: the whole prompt was read' if passed else 'FAIL: the start of the prompt was lost'}")
    print(ollama_ps())
    return passed

if RUN_LONG_CONTEXT_TEST:
    long_context_test()
else:
    print("Skipped. Set RUN_LONG_CONTEXT_TEST = True in step 0, or run: long_context_test()")

## 10. Run your thesis pipeline on this machine

Your code must run **here**, on the Colab machine. Don't expose port 1234 to the internet with tunnels: Colab's rules ban web services and remote proxies on all runtimes.

Your pipeline only needs `base_url = os.environ["LLM_BASE_URL"]` (`http://localhost:1234/v1`), `model = os.environ["LLM_MODEL"]` (`qwen38-unc`) and any API key. Both variables are already set (step 8b).

For a private GitHub repo, store a token as the Colab secret `GITHUB_TOKEN`, then use
`REPO_URL = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/<you>/<repo>.git"`.

**Write each task's result to `OUT_DIR` as soon as it finishes, and skip tasks that already have a result.** A disconnect then costs you one task, not the whole batch.

In [ ]:
REPO_URL = ""       # e.g. "https://github.com/<you>/<thesis-repo>.git"
PIPELINE_CMD = ""   # e.g. f"python -m your_pipeline --out {OUT_DIR}"

if REPO_URL:
    !git clone {REPO_URL} /content/thesis
    !pip -q install -r /content/thesis/requirements.txt
if PIPELINE_CMD:
    !cd /content/thesis && {PIPELINE_CMD}
if not (REPO_URL or PIPELINE_CMD):
    print("Set REPO_URL / PIPELINE_CMD to run your pipeline here.")

## 11. (Optional) Interactive chat, saved to Drive

Each chat is saved after every reply as `.md` (readable) and `.json` (to continue later) in `OUT_DIR/chats`. The thinking is shown as dots and is **not** kept in the history. Run `chat()` for a new chat, or `chat("<file>.json")` to continue one. Type `exit` to stop.

In [ ]:
def chat(continue_from=""):
    chat_dir = os.path.join(OUT_DIR, "chats")
    os.makedirs(chat_dir, exist_ok=True)
    if continue_from:
        chat_path = os.path.join(chat_dir, continue_from)
        with open(chat_path) as f:
            history = json.load(f)["messages"]
        print(f"Continuing {continue_from} ({len(history)} earlier messages)")
    else:
        chat_path = os.path.join(chat_dir, datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + ".json")
        history = []
    md_path = chat_path[:-5] + ".md"

    def save():
        with open(chat_path, "w") as f:
            json.dump({"model": MODEL, "quant": QUANT_SEL, "context": CTX, "messages": history},
                      f, ensure_ascii=False, indent=1)
        with open(md_path, "w") as f:
            f.write(f"# Chat with {MODEL} ({QUANT_SEL}, context {CTX:,})\n\n")
            for m in history:
                who = "You" if m["role"] == "user" else "Model"
                f.write(f"**{who}:**\n\n{m['content']}\n\n---\n\n")

    print(f"Saving to {chat_path}\nType 'exit' to stop.\n")
    while True:
        user = input("You: ")
        if user.strip().lower() in {"exit", "quit"}:
            break
        history.append({"role": "user", "content": user})
        answer, thinking_chars = "", 0
        print("Model: ", end="", flush=True)
        for chunk in client.chat(model=MODEL, messages=history, stream=True, think=True):
            thinking = chunk.message.thinking or ""
            if thinking:
                before = thinking_chars // 2000
                thinking_chars += len(thinking)
                if thinking_chars // 2000 > before:
                    print("·", end="", flush=True)          # one dot per ~2,000 characters of thinking
            if chunk.message.content:
                print(chunk.message.content, end="", flush=True)
                answer += chunk.message.content
        print("\n")
        history.append({"role": "assistant", "content": answer})   # thinking is never stored
        save()
    print("Chat saved:", md_path)

if RUN_CHAT:
    chat()
else:
    print("Chat not started. Set RUN_CHAT = True in step 0, or run: chat()")

## 12. Status, and when you're done

In [ ]:
print(ollama_ps())
print(f"GPU memory used: {gpu_mem_used_mib():,} / {GPU_TOTAL_MIB:,} MiB")
print(f"Endpoint: http://localhost:{PORT}/v1 | model: {MODEL} | context: {CTX:,} | quant: {QUANT_SEL}")
print("Calls logged to:", LOG_PATH)
print("\nWhen finished: Runtime -> Disconnect and delete runtime (stops billing).")
# stop_server()   # uncomment to stop the model server without disconnecting

## Good to know (lessons from testing)

**Speed and cost**
- **Billing:** you pay for the time a GPU is connected, not for generation. Approximate rates: L4 1.71 units/h, A100 40GB 5.40/h, A100 80GB 7.52/h. Your plan includes 50 units/month.
- **L4 speed:** about **13.8 tokens/s** (Q4_K_M), so one attempt of about 27K generated tokens takes roughly half an hour. The A100 is about 3× faster.

**The model**
- **Model storage:** download fresh each session (66 s). Loading the model from Drive took 327 s the first time.
- **Log notice** `model has unused tensor blk.64.… nextn… -- ignoring`: harmless. It's the multi-token-prediction head, which Ollama doesn't use.
- **Endless thinking:** uncensored versions sometimes think until the token budget runs out, especially on prompts a normal model would refuse. Watch for `finish_reason=length` with an empty answer. Use temperature 1.0 (the default here) and a generous `max_tokens`.

**Context and output**
- **Silent truncation:** Ollama drops the **start** of a prompt that's longer than the context, without any error. `llm_call()` warns when `prompt_tokens` gets close to the context size, and step 9 tests for it directly.
- **Changing the context later:** run `load_and_check(<size>)`, then `CTX = <size>`.
- **JSON schema:** the output follows the schema's structure, but still validate its content (`pydantic_validator`). Keep thinking on: turning it off can silently drop the schema on some models. Deeply nested or recursive schemas can degrade the output.

**Rules and housekeeping**
- **No tunnels or port forwarding to your PC.** Colab bans web services and remote proxies on all runtimes. Run your pipeline here (step 10).
- **Keep the browser tab open during long runs.** Background execution is only available with Google AI Ultra.
- **Record for your thesis:** `run_config.json` (GPU, file, quant, context, cache type, Ollama version) and `llm_calls.jsonl` (per-call tokens, time, finish reason).